# E02_CartPoleUnity — Q-learning с дискретизацией

**Урок:** 1.5 курса [cyber-unity-learn](https://rl-cuber-unity-code.com/)
**Что реализуется:** табличный Q-learning над непрерывным наблюдением, разбитым сеткой.
**Ожидаемый результат:** средняя награда **≥ 195** из 200 — классический критерий «CartPole решён».
**Время обучения:** ~9 минут на сид (`QUICK_RUN = True` — около минуты).

## Что нового по сравнению с `E01_GridWorld`

В `E01` состояние было готовым: среда отдавала one-hot вектор, и таблица
индексировалась напрямую. Здесь наблюдение **непрерывно** — четыре
вещественных числа, — и состояний бесконечно много. Табличный метод к такой
среде неприменим, пока не сделан шаг, ради которого и существует урок 1.5:
пространство режется сеткой на конечное число ячеек, и **ячейка** становится
состоянием таблицы.

За это платят: две разные ситуации внутри одной ячейки становятся для агента
неразличимыми. Выбор сетки — компромисс между размером таблицы (её нужно
заполнить опытом) и потерей информации. Отсюда прямой мост к `E03`, где
разбиение не задаётся руками, а обучается вместе с политикой.

## Где живёт сетка

Сетка — часть **алгоритма**, а не среды: Unity отдаёт сырое наблюдение.
Но при экспорте она обязана уехать в граф ONNX (требование 10.7) — иначе
Unity истолкует вход иначе, чем обучение, и агент в редакторе поведёт себя
не так, как в Python. Реализация — `labrl.nets.discretized.DiscretizedQTable`,
и ниже это проверяется числовым паритетом на **сырых** наблюдениях.

## Требования

```powershell
python scripts\build_env.py E02_CartPoleUnity
```


## 1. Окружение

In [1]:
# --- 1. Окружение -------------------------------------------------------
import platform
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "python" / "labrl").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("не найден корень репозитория")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "python"))

import numpy as np
import torch
import mlagents_envs

from labrl.utils.seeding import resolve_device, set_global_seed

SEED = 0
set_global_seed(SEED)
DEVICE = resolve_device("auto")

print(f"python        {platform.python_version()}")
print(f"torch         {torch.__version__}  CUDA {torch.version.cuda}")
print(f"mlagents_envs {mlagents_envs.__version__}")
print(f"устройство    {DEVICE}")
print(f"сид           {SEED}")


python        3.10.11
torch         2.2.1+cu121  CUDA 12.1
mlagents_envs 1.2.0.dev0
устройство    cuda
сид           0


## 2. Конфигурация

`QLearningConfig` — свойства **метода**, `TabularTrainConfig` — свойства
**процесса обучения**.

Про расписание α. Скорость обучения здесь **убывает**, и это не тонкая
настройка, а условие сходимости. Теорема Роббинса–Монро требует убывающего
шага; при постоянном α таблица не сходится, а колеблется вокруг решения
с амплитудой порядка α·|TD-ошибка|. На задаче предсказания это почти
незаметно, на задаче управления — фатально: колебание Q переворачивает
`argmax`, и жадная политика скачет между хорошей и негодной от оценки
к оценке. Измерено на этой самой среде: T-11 в `docs/07_TROUBLESHOOTING.md`.


In [2]:
QUICK_RUN = True   # True — быстрая проверка (~1 мин); False — полное обучение (~9 мин)

from labrl.algos.tabular.q_learning import QLearningConfig
from labrl.train.tabular import TabularTrainConfig
from labrl.utils.schedules import LinearSchedule

TOTAL_STEPS = 15_000 if QUICK_RUN else 150_000

ALGO_CFG = QLearningConfig(
    gamma=0.99,          # горизонт дисконтирования
    learning_rate=0.20,  # стартовое α; далее его задаёт LEARNING_RATE
    initial_q=0.0,       # нейтральная инициализация
)

TRAIN_CFG = TabularTrainConfig(
    total_steps=TOTAL_STEPS,
    eval_every_steps=max(1, TOTAL_STEPS // 10),
    eval_episodes=20,
    success_threshold=195.0,   # успех = эпизод «решён» по критерию CartPole-v0
)

# Расписания НЕ сжимаются вместе с бюджетом: сокращённый прогон должен быть
# началом полного, а не другим экспериментом (T-10 в docs/07_TROUBLESHOOTING.md).
EPSILON = LinearSchedule(start=1.0, end=0.02, decay_steps=60_000)
LEARNING_RATE = LinearSchedule(start=0.20, end=0.01, decay_steps=120_000)

print(f"бюджет: {TOTAL_STEPS} шагов сбора, оценка каждые {TRAIN_CFG.eval_every_steps}")


бюджет: 15000 шагов сбора, оценка каждые 1500


## 3. Подключение среды

In [3]:
from labrl.envs.registry import get as get_env
from labrl.envs.unity_env import open_unity_env
from labrl.envs.vec_unity_env import VecUnityEnv

ENV_ID = "E02_CartPoleUnity"
BUILD = get_env(ENV_ID).build_path
if not BUILD.is_file():
    raise FileNotFoundError(f"нет билда {BUILD}\nСоберите: python scripts/build_env.py {ENV_ID}")

# Физического движка в этой среде нет — динамика интегрируется вручную,
# поэтому time_scale не искажает задачу и ограничен только скоростью процесса.
handle = open_unity_env(ENV_ID, build_path=BUILD, seed=SEED, time_scale=20.0, no_graphics=True)
vec = VecUnityEnv(handle)
obs = vec.reset()
print(handle.describe())
print(f"\nарен (параллельных сред): {vec.num_envs}")


behavior: E02_CartPoleUnity?team=0
наблюдения (1 сенсоров):
  obs_0: shape=(4,)  имя сенсора: 'VectorSensor_size4'
действия: непрерывных 0, дискретные ветки (2,)

арен (параллельных сред): 8


## 4. Инспекция пространств

`assert` сверяет фактическую среду с `ENV_SPEC.md`. Расхождение обязано падать
здесь, а не проявляться загадочными результатами через час обучения.


In [4]:
OBS_DIM = vec.single_obs_dim
NUM_ACTIONS = int(vec.action_spec.discrete_branches[0])

print(f"obs_shapes:        {vec.obs_shapes}")
print(f"obs_dim:           {OBS_DIM}")
print(f"дискретные ветки:  {tuple(vec.action_spec.discrete_branches)}")
print(f"непрерывных:       {vec.action_spec.continuous_size}")
print(f"\nпример наблюдения (x, ẋ, θ, θ̇): {np.array2string(obs[0][0], precision=4)}")

# Значения из строки-контракта ENV_SPEC.md:
# <!-- validator: obs_size=4; discrete_branches=2; continuous_size=0; max_step=200 -->
assert OBS_DIM == 4, f"obs_dim {OBS_DIM} != 4 (ENV_SPEC.md)"
assert NUM_ACTIONS == 2, f"веток {NUM_ACTIONS} != 2 (ENV_SPEC.md)"
assert vec.action_spec.continuous_size == 0
print("\nразмерности совпадают с ENV_SPEC.md")


obs_shapes:        [(4,)]
obs_dim:           4
дискретные ветки:  (2,)
непрерывных:       0

пример наблюдения (x, ẋ, θ, θ̇): [ 0.0316  0.0491  0.0064 -0.0088]

размерности совпадают с ENV_SPEC.md


## 5. Сетка дискретизации — «архитектура» этого примера

Здесь нет нейросети; роль архитектуры играет **сетка**, и менять её нужно
именно в этой ячейке.

Что стоит за числами:

* **углу θ и угловой скорости θ̇ отдано вчетверо больше ячеек**, чем положению
  тележки: шест падает за десятки шагов, а тележка доезжает до границы
  за сотни. Разрешение по θ важнее;
* **края сетки разомкнуты**: значение ниже первой границы попадает в нулевую
  ячейку, выше последней — в последнюю. Поэтому границы не обязаны покрывать
  экстремумы — достаточно покрыть область, где нужна детализация;
* **размер таблицы** = произведение числа ячеек. Каждую пару (состояние,
  действие) нужно посетить много раз, поэтому вчетверо более подробная сетка
  требует примерно вчетверо большего бюджета.

Попробуйте `bins=[1, 1, 6, 12]` — классический вариант из учебников, который
вовсе игнорирует положение тележки.


In [5]:
from labrl.envs.state_encoders import BoxDiscretizer

GRID = BoxDiscretizer.uniform(
    #        x      ẋ      θ         θ̇
    lows=  [-2.4, -3.0, -0.2095, -2.5],
    highs= [ 2.4,  3.0,  0.2095,  2.5],
    bins=  [3, 3, 8, 8],
)

print(GRID.describe())
print(f"\nразмер таблицы Q: ({GRID.num_states}, {NUM_ACTIONS}) = "
      f"{GRID.num_states * NUM_ACTIONS} чисел")

# Как выглядит перевод наблюдения в индекс состояния.
print(f"\nнаблюдение {np.array2string(obs[0][0], precision=3)} -> "
      f"ячейки {GRID.cell(obs[0][:1])[0]} -> состояние {GRID.index(obs[0][:1])[0]}")


состояний: 576 = 3 × 3 × 8 × 8
  измерение 0: 3 ячеек, границы [-0.800, +0.800]
  измерение 1: 3 ячеек, границы [-1.000, +1.000]
  измерение 2: 8 ячеек, границы [-0.157, -0.105, -0.052, +0.000, +0.052, +0.105, +0.157]
  измерение 3: 8 ячеек, границы [-1.875, -1.250, -0.625, +0.000, +0.625, +1.250, +1.875]

размер таблицы Q: (576, 2) = 1152 чисел

наблюдение [ 0.032  0.049  0.006 -0.009] -> ячейки [1 1 4 3] -> состояние 291


## 6. Теория и математика метода

Правило обновления то же, что в `E01`, — меняется только то, откуда берётся
индекс состояния:

$$
y_t = r_t + \gamma\,(1-\mathrm{terminated}_t)\,\max_{a'} Q(s_{t+1}, a')
$$
$$
Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha_t\,\bigl(y_t - Q(s_t,a_t)\bigr)
$$

где $s_t = \mathrm{grid}(o_t)$ — номер ячейки, в которую попало наблюдение.

Про множитель $(1-\mathrm{terminated})$. Будущее обнуляется **только** при
истинном падении шеста. Обрыв по `MaxStep` — не терминальное состояние: шест
там стоял прямо, и ценность этого состояния высока. Смешать эти случаи
значило бы научить агента, что дожить до конца эпизода так же плохо, как
уронить шест. Именно в этой среде ошибка была бы особенно наглядной: все
успешные эпизоды заканчиваются обрывом по времени.

Про $\alpha_t$: см. раздел 2 — убывание требуется условием сходимости.

Ниже — исходный код обновления, тот самый, что выполняется при обучении.


In [6]:
import inspect

from labrl.algos.tabular.q_learning import QLearning

print(inspect.getsource(QLearning.update))


    def update(self, batch: Transition) -> dict[str, float]:
        """Единственное место, где меняется таблица Q (требование 8.7).

        Обработка повторов внутри батча. Если две арены дали переход из одной
        и той же пары (s, a), наивное векторное присваивание применит только
        последнее обновление — остальные потеряются. Поэтому приращения
        суммируются через ``np.add.at``: это эквивалентно последовательному
        применению всех переходов батча с одинаковым «старым» Q, что и есть
        стандартная семантика батчевого табличного обновления.

        Returns:
            Метрики шага обучения для TensorBoard.
        """
        state = np.asarray(batch.state, dtype=np.int64)
        action = np.asarray(batch.action, dtype=np.int64)
        reward = np.asarray(batch.reward, dtype=np.float64)
        next_state = np.asarray(batch.next_state, dtype=np.int64)
        terminated = np.asarray(batch.terminated, dtype=bool)

        if state.size == 0:
            

## 7. Цикл обучения

Цикл импортируется из `labrl.train.tabular`, а не копируется сюда: иначе
учебная и рабочая версии со временем разойдутся.


In [7]:
from labrl.logging.run_dir import create_run_dir
from labrl.logging.tb_logger import TBLogger
from labrl.train.tabular import train_q_learning

algo = QLearning(num_states=GRID.num_states, num_actions=NUM_ACTIONS,
                 cfg=ALGO_CFG, encoder=GRID)

run = create_run_dir(ENV_ID, "qlearning", SEED)
run.write_env_info({**handle.env_info(), "num_envs": vec.num_envs, "seed": SEED,
                    "total_steps": TOTAL_STEPS, "quick": QUICK_RUN})
logger = TBLogger(run.tb)

print(f"каталог прогона: {run.root}\n")

result = train_q_learning(
    vec=vec, algo=algo, epsilon_schedule=EPSILON, logger=logger,
    cfg=TRAIN_CFG, seed=SEED, lr_schedule=LEARNING_RATE,
    on_eval=lambda step, r, s: print(f"  шаг {step:>7}: награда {r:>7.2f}, решено {s:.0%}"),
)

missing = logger.missing_required_tags()
logger.dump_metrics(run.metrics_json, extra={"seed": SEED, "missing_required_tags": list(missing)})
logger.close()

visited = int((np.abs(algo.q).sum(axis=1) > 0).sum())
print(f"\nэпизодов: {len(result.episode_returns)}, переходов: {result.transitions}")
print(f"посещено состояний: {visited} из {GRID.num_states} ({visited / GRID.num_states:.0%})")
print(f"время: {result.wall_time:.1f} с")
print(f"обязательные теги схемы 11.2 без данных: {missing or 'нет'}")


каталог прогона: C:\unity-ml-agents-lab\results\E02_CartPoleUnity\qlearning\20260815-200455_seed0



  шаг    1500: награда   88.80, решено 0%


  шаг    3000: награда  112.35, решено 0%


  шаг    4500: награда  113.60, решено 0%


  шаг    6000: награда  200.00, решено 100%


...

  шаг    9000: награда  152.10, решено 15%


  шаг   10500: награда  159.90, решено 15%


  шаг   12000: награда  146.90, решено 10%


  шаг   13500: награда  165.70, решено 15%


  шаг   15000: награда  160.00, решено 25%

эпизодов: 4161, переходов: 120000
посещено состояний: 225 из 576 (39%)
время: 82.3 с
обязательные теги схемы 11.2 без данных: нет


## 8. Оценка и кривая обучения

Правый график — доля посещённых ячеек по паре (θ, θ̇). Непосещённые ячейки —
это состояния, о которых таблица не знает ничего; если агент в них попадёт,
он выберет действие 0 просто потому, что при равных нулях `argmax` берёт
первое. Пустоты по краям — норма: агент туда не заходит. Пустоты в центре
означали бы недостаточный бюджет.


In [8]:
import matplotlib
matplotlib.use("Agg")   # ноутбук должен исполняться и без графического бэкенда
import matplotlib.pyplot as plt

returns = np.array(result.episode_returns)
window = max(1, len(returns) // 50)
smoothed = np.convolve(returns, np.ones(window) / window, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

axes[0].plot(smoothed)
axes[0].set_title(f"Награда за эпизод (скользящее среднее по {window})")
axes[0].set_xlabel("эпизод"); axes[0].set_ylabel("шагов прожито"); axes[0].grid(alpha=0.3)

steps = [s for s, _, _ in result.eval_history]
rewards = [r for _, r, _ in result.eval_history]
axes[1].plot(steps, rewards, marker="o")
axes[1].axhline(195, color="k", ls="--", lw=1, label="порог «решено»")
axes[1].set_title("Eval/Mean Reward (жадная политика)")
axes[1].set_xlabel("шаг сбора"); axes[1].grid(alpha=0.3); axes[1].legend()

# Посещённость по (θ, θ̇): усредняем по двум измерениям тележки.
visited_cells = (np.abs(algo.q).sum(axis=1) > 0).reshape(GRID.bins_per_dim)
heat = visited_cells.mean(axis=(0, 1))
im = axes[2].imshow(heat.T, origin="lower", cmap="viridis", vmin=0, vmax=1, aspect="auto")
axes[2].set_title("Доля посещённых ячеек")
axes[2].set_xlabel("ячейка θ"); axes[2].set_ylabel("ячейка θ̇")
fig.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()

print(f"итоговая оценка: награда {result.last_eval_reward:.2f}, решено {result.last_eval_success:.0%}")

THRESHOLD = 195.0 if not QUICK_RUN else 100.0   # блок success_criteria конфига
assert result.last_eval_reward >= THRESHOLD, (
    f"не достигнут критерий приёмки: {result.last_eval_reward:.2f} < {THRESHOLD}")


итоговая оценка: награда 160.00, решено 25%


C:\Users\-\AppData\Local\Temp\ipykernel_22728\2643055942.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Экспорт в ONNX и верификация

Ключевой момент примера. `policy_module()` возвращает **не** голую таблицу,
а модуль, который сначала применяет сетку, а потом выбирает строку:

```
obs (B, 4) -> номер ячейки по каждому измерению -> индекс состояния -> Q (B, 2)
```

Всё это выражено операциями opset 9 — того самого, с которым ML-Agents
экспортирует модели. Числовой паритет ниже считается на **сырых**
наблюдениях, поэтому он проверяет и таблицу, и сетку разом. Если бы
дискретизация осталась в Python, проверка это и показала бы.


In [9]:
from labrl.export.onnx_export import ActionSpecLite, export_policy_to_onnx
from labrl.export.onnx_verify import verify_onnx_model

# Наблюдения из среды под текущей политикой.
collected, obs_now = [], vec.reset()
while sum(len(c) for c in collected) < 64:
    collected.append(obs_now[0].copy())
    actions = algo.greedy_action(GRID.index(obs_now[0]))[:, None].astype(np.int32)
    obs_now = vec.step(actions).obs
sample = [np.concatenate(collected, axis=0)[:64]]

spec = ActionSpecLite(discrete_branches=(NUM_ACTIONS,))
policy = algo.policy_module()
print(f"экспортируемый модуль: {type(policy).__name__}\n")

onnx_path = export_policy_to_onnx(policy, spec, vec.obs_shapes, run.onnx / "policy.onnx")
verification = verify_onnx_model(onnx_path, policy, spec, vec.obs_shapes, sample_obs=sample)
print(verification.report())
verification.raise_if_failed()

model_dest = (REPO_ROOT / "unity" / "MLAgentsLab" / "Assets" / "Envs" / ENV_ID /
              "Models" / f"{ENV_ID}_qlearning.onnx")
model_dest.parent.mkdir(parents=True, exist_ok=True)
model_dest.write_bytes(onnx_path.read_bytes())
print(f"\nмодель скопирована в проект Unity: {model_dest.relative_to(REPO_ROOT)}")

vec.close()


экспортируемый модуль: DiscretizedQTable



OK    структура (onnx.checker)
OK    opset — ожидался 9, в модели 9
OK    имена входов — лишние=[], отсутствуют=[]
OK    имена выходов — лишние=[], отсутствуют=[]
OK    тип выхода discrete_actions — ожидался целочисленный тензор, получен elem_type=7 (int64)
OK    тип выхода deterministic_discrete_actions — ожидался целочисленный тензор, получен elem_type=7 (int64)
OK    прогон onnxruntime, батч 1
OK    прогон onnxruntime, батч 64
OK    константа version_number — ожидалось [3.0], получено [3.0]
OK    константа memory_size — ожидалось [0.0], получено [0.0]
OK    константа discrete_action_output_shape — ожидалось [2.0], получено [2.0]
OK    числовой паритет: deterministic_discrete_actions — max|ONNX-PyTorch| = 0.000e+00, допуск 1e-04
OK    диапазон дискретной ветки 0 — значения [0, 1], допустимо целые [0, 2)
ИТОГ: ПРОЙДЕНО (13/13)

модель скопирована в проект Unity: unity\MLAgentsLab\Assets\Envs\E02_CartPoleUnity\Models\E02_CartPoleUnity_qlearning.onnx


## 10. Возврат в Unity

### Автоматически (то, чем проверяется приёмка по 10.6)

```powershell
python scripts\check_inference.py --config configs\E02_CartPoleUnity__qlearning.yaml --python-reward 198.0
```

Скрипт соберёт билд с назначенной моделью и `Behavior Type = Inference Only`,
прогонит 20 эпизодов и сравнит награду с Python-оценкой. Порог — 0.8 от неё.

Среда детерминирована полностью (физического движка нет), поэтому здесь
ожидается совпадение, а не «примерно столько же»: расхождение означало бы
реальную ошибку в графе или в порядке наблюдений.

### Вручную (чтобы увидеть глазами)

1. Открыть `unity/MLAgentsLab` в Unity 6000.5.4f1.
2. Открыть `Assets/Envs/E02_CartPoleUnity/Scenes/E02_CartPoleUnity.unity`.
3. Выделить модель в Project → **Tools → RL → Check ONNX Contract**.
4. У агента: `Model` = импортированная модель, `Behavior Type` = `Inference Only`,
   галочка `Deterministic Inference`.
5. Play: шест стоит вертикально, тележка мелко подрагивает возле центра.
   Восемь арен ведут себя по-разному — у них разные стартовые состояния.

### Ручная проверка среды с клавиатуры

`Behavior Type` = `Heuristic Only`, клавиши **←/→** толкают тележку.
Удержать шест руками почти невозможно — это нормально и хорошо показывает,
почему задача считается нетривиальной.
